# RetailRocket — Exploratory Data Analysis

**Dataset:** [RetailRocket E-Commerce Dataset](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset)  
**Goal:** Understand the structure, distributions, and patterns in the event log before building a purchase-prediction model.

## Contents
1. Load & inspect data  
2. Event distribution  
3. Visitor behaviour  
4. Time patterns  
5. Key findings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.path.join('..', 'src'))

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load & Inspect

In [ ]:
DATA_DIR = os.path.join('..', 'data')

df = pd.read_csv(os.path.join(DATA_DIR, 'events.csv'))
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
print('=== Missing values ===')
print(df.isnull().sum())
print()
print('Note: transactionid is only set for transaction events — NaN for views/addtocart is expected.')

In [ ]:
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Unique visitors : {df['visitorid'].nunique():,}")
print(f"Unique items    : {df['itemid'].nunique():,}")

## 2. Event Distribution

In [ ]:
event_counts = df['event'].value_counts()
event_pct    = (event_counts / len(df) * 100).round(2)

summary = pd.DataFrame({'count': event_counts, 'pct (%)': event_pct})
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(event_counts.index, event_counts.values, color=['#4C72B0', '#DD8452', '#55A868'])
axes[0].set_title('Event counts')
axes[0].set_ylabel('Number of events')
for i, (name, count) in enumerate(event_counts.items()):
    axes[0].text(i, count * 1.01, f'{count:,}', ha='center', fontsize=9)

axes[1].pie(event_pct.values, labels=event_pct.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452', '#55A868'], startangle=90)
axes[1].set_title('Event share')

plt.tight_layout()
plt.show()

**Finding:** The funnel is extremely skewed — 96.7 % views, 2.5 % add-to-cart, 0.8 % transactions. This class imbalance will be the main challenge for modelling.

## 3. Visitor Behaviour

In [ ]:
events_per_visitor = df.groupby('visitorid').size()

print('Events per visitor:')
print(events_per_visitor.describe().round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
clipped = events_per_visitor.clip(upper=50)
ax.hist(clipped, bins=50, color='#4C72B0', edgecolor='white')
ax.set_xlabel('Events per visitor (capped at 50)')
ax.set_ylabel('Number of visitors')
ax.set_title('Distribution of events per visitor')
plt.tight_layout()
plt.show()

In [ ]:
# Visitors who purchased vs. never purchased
buyers = df[df['event'] == 'transaction']['visitorid'].unique()
print(f'Visitors who purchased at least once : {len(buyers):,}')
print(f'Visitors who never purchased          : {df["visitorid"].nunique() - len(buyers):,}')
print(f'Purchase conversion rate (visitors)   : {len(buyers) / df["visitorid"].nunique() * 100:.2f} %')

## 4. Time Patterns

In [ ]:
df['hour']    = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.day_name()
df['week']    = df['timestamp'].dt.isocalendar().week.astype(int)

WEEKDAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# By hour
hourly = df.groupby('hour').size()
axes[0].plot(hourly.index, hourly.values, marker='o', color='#4C72B0')
axes[0].set_title('Events by hour of day')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('Events')
axes[0].set_xticks(range(0, 24, 2))

# By weekday
daily = df.groupby('weekday').size().reindex(WEEKDAY_ORDER)
axes[1].bar(range(7), daily.values, color='#DD8452')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels([d[:3] for d in WEEKDAY_ORDER])
axes[1].set_title('Events by weekday')
axes[1].set_ylabel('Events')

plt.tight_layout()
plt.show()

## 5. Key Findings

| # | Finding |
|---|---|
| 1 | **2,756,101 events** from **1,407,580 unique visitors** over ~4.5 months (May–Sep 2015) |
| 2 | Event funnel: **96.7 % view → 2.5 % add-to-cart → 0.8 % transaction** |
| 3 | Most visitors are one-time or low-engagement (highly right-skewed events-per-visitor distribution) |
| 4 | Only **~1 %** of visitors ever complete a purchase |
| 5 | Peak activity: **midday hours**, weekday pattern shows weekday-heaviness |
| 6 | `transactionid` is only non-null for transaction events — no data quality issues otherwise |

**Next step → notebook 02: Sessionize and build tabular features**